# Retrospective Election Survey Classification

**Recruiter-facing end-to-end analysis · Historical binary classification · Python 3.12/3.13**

> The calibrated random forest reaches retrospective test ROC-AUC 0.907 and balanced accuracy 0.800; this is respondent classification, not polling.

## Executive summary

**Objective:** Evaluate respondent-level party classification while explicitly separating it from representative polling or forecasting.

**Data:** 1,525 historical survey rows with vote, age, leader ratings, economic assessments, Europe attitudes, knowledge, and gender.

**Verified result:** The calibrated random forest reaches retrospective test ROC-AUC 0.907 and balanced accuracy 0.800; this is respondent classification, not polling.

**Decision supported:** Understand classification signal and uncertainty without making live-election claims.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A survey-methodology analyst.

**Decision:** Understand classification signal and uncertainty without making live-election claims.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '07-election-exit-poll-prediction'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 07-election-exit-poll-prediction


## 4. Data provenance and scope

Bundled in the original repository; survey organization, weighting, sampling design, and license are undocumented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

              file  size_mb           sha256
election_data.xlsx    0.078 af2bb5a60c1923c5


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


election_data.xlsx: 10 columns
 Unnamed: 0   vote  age  economic.cond.national  economic.cond.household  Blair  Hague  Europe  political.knowledge gender
          1 Labour   43                       3                        3      4      1       2                    2 female
          2 Labour   36                       4                        4      4      4       5                    2   male
          3 Labour   35                       4                        4      5      2       3                    2   male
          4 Labour   24                       4                        2      2      1       4                    0 female
          5 Labour   41                       2                        2      1      1       6                    2   male


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 113 lines
Functions: run_analysis


## 7. Methodology and hypotheses

Stratified CV, dummy/logistic/LDA/KNN/SVM/random-forest comparison, probability calibration, untouched test evaluation, permutation importance, and age/gender sensitivity tables.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_07_election_exit_poll_prediction", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 46.56 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (10 fields)
                 column  dtype  missing_count  missing_percent  unique_values  constant
             Unnamed: 0  int64              0              0.0           1525     False
                   vote object              0              0.0              2     False
                    age  int64              0              0.0             70     False
 economic.cond.national  int64              0              0.0              5     False
economic.cond.household  int64              0              0.0              5     False
                  Blair  int64              0              0.0              5     False
                  Hague  int64              0              0.0              5     False
                 Europe  int64              0              0.0             11     False
    political.knowledge  int64              0              0.0              4     False
                 gender object              0              0.0              2     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'model_comparison.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'The calibrated random forest reaches retrospective test ROC-AUC 0.907 and balanced accuracy 0.800; this is respondent classification, not polling.')

Primary evidence: model_comparison.csv, shape=(6, 5)
                       model  cv_roc_auc_mean  cv_roc_auc_std  cv_balanced_accuracy_mean  cv_f1_mean
               random_forest           0.8821          0.0258                     0.8030      0.8632
linear_discriminant_analysis           0.8811          0.0169                     0.7832      0.8789
      support_vector_machine           0.8797          0.0295                     0.8139      0.8598
         logistic_regression           0.8790          0.0174                     0.8078      0.8549
                         knn           0.8723          0.0279                     0.7758      0.8771
              dummy_majority           0.5000          0.0000                     0.5000      0.8216

Verified result:
The calibrated random forest reaches retrospective test ROC-AUC 0.907 and balanced accuracy 0.800; this is respondent classification, not polling.


## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "training_rows": 1143,
  "untouched_test_rows": 382,
  "selection": "5-fold stratified CV on training only",
  "selection_metric": "ROC-AUC",
  "random_seed": 42
}


## 12. Visual evidence

### Election Model Evidence

![election_model_evidence](../reports/figures/election_model_evidence.png)

### Model Comparison

![model_comparison](../reports/figures/model_comparison.png)

## 13. Business interpretation

The calibrated random forest reaches retrospective test ROC-AUC 0.907 and balanced accuracy 0.800; this is respondent classification, not polling.

The correct action is to use this result as evidence for **Understand classification signal and uncertainty without making live-election claims.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

The undocumented sampling and weighting design prevents representative election inference or future-election claims.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                   artifact  size_kb       sha256
reports/figures/election_model_evidence.png    180.3 e742ad64b7da
       reports/figures/model_comparison.png     72.2 76a7f653037a
                       reports/metrics.json      4.4 7b2145184ea2
            reports/tables/data_quality.csv      0.4 b120c6c909f3
        reports/tables/model_comparison.csv      0.6 9f93fdc73513
  reports/tables/permutation_importance.csv      0.5 87730b55925c
    reports/tables/subgroup_sensitivity.csv      0.3 97adebde84f3


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed evaluate respondent-level party classification while explicitly separating it from representative polling or forecasting. using stratified cv, dummy/logistic/lda/knn/svm/random-forest comparison, probability calibration, untouched test evaluation, permutation importance, and age/gender sensitivity tables. The final verified conclusion is: **The calibrated random forest reaches retrospective test ROC-AUC 0.907 and balanced accuracy 0.800; this is respondent classification, not polling.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/07-election-exit-poll-prediction/src/analysis.py
python scripts/execute_notebooks.py --project 07-election-exit-poll-prediction
```